# Spotify Lab v3: Simple Vector Recommendations

**Name(s):**

You are a new data engineer at Spotify. When a listener asks for songs with similar content, we want to search a library of songs and return useful recommendations.

In this lab, every song is represented as a vector of lyric word counts. If the word "love" appears 12 times in a song, then the song has coordinate $x_{love}=12$ for that word. Songs with similar word-count patterns should have vectors that look similar.

The main files are:

- `./data/song_vectors.csv.gz`: one row per song
- `./data/vocabulary.csv`: the 800 word columns used in the vectors
- `./data/top_40_songs_by_genre.csv`: 40 popular songs from each genre, used as a benchmark

The genre labels are useful for evaluation, but they are not perfect ground truth. Some songs cross genres, some labels are debatable, and real recommendation systems would also use audio features, listener behavior, and human judgments.

The main idea of the lab:

**Dot product rewards shared word-count mass. Cosine similarity corrects for vector length. TF-IDF changes which words matter.**

## What You Will Submit

Complete the notebook and answer the yellow prompts in your own words. Your answers should focus on what each scoring rule rewards and whether the recommendations seem useful.

At the end, write a **short 2-4 paragraph report**. State which method you would recommend for this lyric-only prototype, use examples from the rankings, and explain one important limitation of using lyric word counts alone.

## Setup

We will use only `pandas`, `numpy`, and `seaborn`.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

songs = pd.read_csv("./data/song_vectors.csv.gz")
vocab = pd.read_csv("./data/vocabulary.csv")
top_40 = pd.read_csv("./data/top_40_songs_by_genre.csv")

word_cols = vocab["word"].tolist()
meta_cols = [col for col in songs.columns if col not in word_cols]

X = songs[word_cols].to_numpy(dtype=float)

print("songs:", songs.shape)
print("word-count matrix:", X.shape)
print("top-40 benchmark:", top_40.shape)

songs: (8000, 816)
word-count matrix: (8000, 800)
top-40 benchmark: (560, 6)


In [2]:
songs[meta_cols].head()

,track_id,song_id,artist_id,artist_name,song_title,release,release_year,duration,artist_familiarity,artist_hotttnesss,genre,top_tags,n_words,nnz_words,l2_norm,is_test
0,TRAAPIV128F1493132,SOJHKYL12A6D4FA0AC,AR56P361187B9AC4DB,Gary Moore,What Are We Here For,Dark Days In Paradise,1997,345.93914,0.761362,0.467111,blues,blues; classic rock; rock; hard rock; blues rock,218,80,37.376463,0
1,TRAAWOR128F92DF3C8,SOBJYXZ12A8AE45FF1,ARY80281187FB3E380,Sandi Thom,Wounded Hearts,The Pink & The Lily,2008,205.21751,0.655359,0.405031,blues,blues,287,103,39.698866,0
2,TRABNZS128F932C6E8,SOCOZZE12A813568F7,AROWOZ41187FB5B535,G. Love & Special Sauce,Willow Tree,Yeah_ It's That Easy,1997,206.57587,0.717235,0.548694,blues,blues; chill; alternative; rock; jazz,211,64,33.090784,0
3,TRABOIH128F9300B06,SOHYOTB12AF72AA1D7,AR94ZOI1187FB46BDA,Willie Dixon,That's My Baby,Very Best Of The Blues,1989,208.90077,0.579533,0.429834,blues,delta blues; BLUEZZZ; blues; freedom; Willie D...,164,76,27.477263,0
4,TRABYAR128F931B1A4,SOVKUOE12A58A8066E,AR7BMMV1187FB5B2D7,Robben Ford,Something For The Pain,Blue Moon,2002,298.37016,0.604820,0.383877,blues,blues; Robben Ford; vinnie colaiuta; jimmy earl,187,75,27.604347,0


## Part 1: Inspect One Song Vector

Each row of `X` is one song vector. We will start with a single query song so the scoring rules are easier to see.

In [3]:
query_row = 0
query_track_id = songs.loc[query_row, "track_id"]
query = X[query_row]

songs.loc[[query_row], ["artist_name", "song_title", "genre", "n_words", "nnz_words"]]

,artist_name,song_title,genre,n_words,nnz_words
0,Gary Moore,What Are We Here For,blues,218,80


In [4]:
word_counts = pd.DataFrame({
    "word": word_cols,
    "count": query.astype(int),
})

word_counts.query("count > 0").sort_values("count", ascending=False).head(15)

,word,count
748,we,27
31,are,10
245,for,9
475,of,7
336,it,6
753,what,6
306,here,6
295,have,6
489,our,5
676,through,5


<div class="alert alert-block alert-warning">

What are the most common words in the query song? Do they look like words that would be useful for recommending similar songs? Why or why not?

</div>

## Part 2: Dot Product

The dot product is:

$$
x \cdot y = \sum_j x_j y_j
$$

It gets large when two songs use many of the same words many times. That can be useful, but it can also reward long or repetitive songs.

In [ ]:
def top_matches(scores, query_track_id, k=10, query_artist_id=None):
    results = songs[meta_cols].copy()
    results["score"] = scores

    results = results[results["track_id"] != query_track_id]
    if query_artist_id is not None:
        results = results[results["artist_id"] != query_artist_id]

    return results.sort_values("score", ascending=False).head(k)

In [ ]:
dot_scores = X @ query
dot_matches = top_matches(dot_scores, query_track_id, k=10)

dot_matches[["artist_name", "song_title", "genre", "n_words", "score"]]

In [ ]:
plot_data = dot_matches.copy()
plot_data["song"] = plot_data["artist_name"] + " - " + plot_data["song_title"]

ax = sns.barplot(data=plot_data, y="song", x="score")
ax.set(title="Dot product top matches", xlabel="dot product", ylabel="")

<div class="alert alert-block alert-warning">

Look at the dot-product matches. Do they seem similar to the query song, or does the score seem to reward song size/repetition? Use at least one specific song from the table in your answer.

</div>

## Part 3: Cosine Similarity

Cosine similarity divides the dot product by the lengths of the two vectors:

$$
\cos(\theta) = \frac{x \cdot y}{\|x\|\|y\|}
$$

This compares the direction of the word-count profiles. A long song does not automatically win just because it has more words.

In [ ]:
def cosine_scores(matrix, query_vector):
    numerators = matrix @ query_vector
    denominators = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vector)

    return np.divide(
        numerators,
        denominators,
        out=np.zeros(len(numerators)),
        where=denominators != 0,
    )

In [ ]:
cos_scores = cosine_scores(X, query)
cos_matches = top_matches(cos_scores, query_track_id, k=10)

cos_matches[["artist_name", "song_title", "genre", "n_words", "score"]]

In [ ]:
plot_data = cos_matches.copy()
plot_data["song"] = plot_data["artist_name"] + " - " + plot_data["song_title"]

ax = sns.barplot(data=plot_data, y="song", x="score")
ax.set(title="Cosine similarity top matches", xlabel="cosine similarity", ylabel="")

<div class="alert alert-block alert-warning">

Compare the dot-product matches to the cosine matches. Which method looks more like a similar-song engine? Which method looks more like it is rewarding raw word-count mass?

</div>

## Part 4: TF-IDF

Some words appear in many songs. Those words can dominate similarity scores even if they do not tell us much about the song.

TF-IDF changes the word-count matrix in two steps:

1. **Term frequency:** `np.log1p(X)` makes each additional repeated use of a word count less than the previous one. This reduces the marginal effect of repetition within a song.
2. **Inverse document frequency:** words that appear in many songs get smaller weights, and words that appear in fewer songs get larger weights.

Then we use the transformed matrix for recommendations.

In [ ]:
TF = np.log1p(X)

document_frequency = (X > 0).sum(axis=0)
IDF = np.log(len(songs) / document_frequency)

TFIDF = TF * IDF

idf_table = pd.DataFrame({
    "word": word_cols,
    "document_frequency": document_frequency,
    "idf": IDF,
})

In [ ]:
idf_table.sort_values("idf").head(10)

In [ ]:
idf_table.sort_values("idf", ascending=False).head(10)

<div class="alert alert-block alert-warning">

What do you notice about the lowest-IDF words and highest-IDF words? Why might this matter for a recommendation system?

</div>

In [ ]:
query_tfidf = TFIDF[query_row]

tfidf_dot_scores = TFIDF @ query_tfidf
tfidf_dot_matches = top_matches(tfidf_dot_scores, query_track_id, k=10)

tfidf_dot_matches[["artist_name", "song_title", "genre", "n_words", "score"]]

In [ ]:
tfidf_cos_scores = cosine_scores(TFIDF, query_tfidf)
tfidf_cos_matches = top_matches(tfidf_cos_scores, query_track_id, k=10)

tfidf_cos_matches[["artist_name", "song_title", "genre", "n_words", "score"]]

<div class="alert alert-block alert-warning">

Compare raw dot product, raw cosine, TF-IDF dot product, and TF-IDF cosine. Which recommendation list seems best for this query song? Give one specific example.

</div>

## Part 5: Benchmark with Genre

Evaluate all four methods on the Top-40 songs.

For each query song, we will:

1. score every possible match,
2. remove the query song itself,
3. remove songs by the same artist,
4. keep the top two matches,
5. record whether each match has the same genre as the query.

The final score is **precision@2**: the share of recommendation slots that match the query genre.

<div class="alert alert-block alert-warning">

Which method has the best precision@2? Does that agree with your judgment from the single-query examples, or does the benchmark change your view?

</div>

## Part 6: Test an Adjustment

A recommendation system is a design choice. Investigate your previous results, think of a promising change to make in the system, and compare it to the pevious methods. Did the adjustment improve results? (It's fine if it didn't.)

## Final Report

Write a short report on your results in a document file:

1. Which method would you recommend for this lyric-only prototype?
2. What evidence from the tables or precision@2 scores supports your choice?
3. Give at least two concrete song examples from the recommendations.
4. What is one important limitation of lyric word counts alone? What other data would be helpful, and how would you use it?